<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="04_low_level_mcp.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_opencode_intro.ipynb">1</a>
        <a href="02_inference_endpoint.ipynb">2</a>
        <a href="03_introduction_mcp.ipynb">3</a>
        <a href="04_low_level_mcp.ipynb">4</a>
        <a >5</a>
        <a href="06_nemo_agent_toolkit.ipynb">6</a>
        <a href="07_challenge.ipynb">7</a>
        <a href="bonus_challenge/08_bonus_challenge.ipynb">8</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="06_nemo_agent_toolkit.ipynb">Next Notebook</a></span>
</div>

## Learning objectives

By the end of this notebook, you will be able to:
- Define LangGraph State schemas and build chatbot workflows using StateGraph, nodes, and edges
- Connect NVIDIA NIM endpoints as the LLM backend using the `nvidia` model provider
- Stream graph responses using `graph.stream()` for real-time output
- Implement structured output with Pydantic models for parseable LLM responses

## Setup Environment 

In the first notebook, we learned how to set up our generated `NVIDIA API KEY`. As a requirement for this notebook, you must set up the key as enviroment variable `NVIDIA_API_KEY` to pull the NIMs docker images of your choice. If you haven't gotten your key, please visit the NVIDIA NIMs API [homepage](https://build.nvidia.com/explore/discover) and generate your API Key. Please run the cell below, input your `NVIDIA API KEY` in the display textbox, and press the enter key on your keyboard.

In [ ]:
import os
import getpass

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key

## Introduction to LangGraph

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages

In [ ]:
from langchain.chat_models import init_chat_model

This creates a LangChain chat model connected to NVIDIA NIM. The `init_chat_model()` function handles all the configuration automatically—just specify the model ID and provider, and you're ready to start generating responses.

In [ ]:
MODEL_ID = 'nvidia/nemotron-3-nano-30b-a3b'
llm = init_chat_model(model=MODEL_ID, model_provider="nvidia",max_tokens=32)

In [ ]:
# Comment this out if you are using the local endpoint with right local port
# LOCAL_CONTAINER_PORT = 11579
# llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(CONTAINER_PORT), model="nvidia/nemotron-3-nano-30b-a3b")

In [ ]:
# llm.get_available_models()

In this section, we'll construct a simple agentic workflow using LangGraph's StateGraph. Here's what we'll do:

1. Create a `StateGraph` with the state schema
2. Add nodes using `add_node(name, function)`
3. Add edges using `add_edge(source, target)`
4. Compile the graph before execution

In [ ]:
class State(TypedDict):
    """
    Graph state schema.
    - messages: List of conversation messages with automatic append behavior
    """
    messages: Annotated[list, add_messages]

The State holds the conversation history using the `add_messages` reducer, which automatically appends new messages to the list.

Nodes are Python functions that receive state, perform actions, and return updated state.

In [ ]:
def chatbot(state: State):
    """
    Chatbot node that invokes the LLM with conversation history.
    Returns updated state with the assistant's response.
    """
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)

# Add the chatbot node
graph_builder.add_node("chatbot", chatbot)

# Connect START -> chatbot (entry point)
graph_builder.add_edge(START, "chatbot")

# Compile the graph
graph = graph_builder.compile()

### Visualize the Graph

Before running the graph, render the compiled LangGraph workflow as a Mermaid diagram. This provides a quick visual check that the `START` node connects to the `chatbot` node as expected.

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

Use `graph.invoke()` to get synchronous complete responses

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "What is chicago known for?"}]})

Use `graph.stream()` to get synchronous token-by-token responses, improving user experience.

In [ ]:
def stream_graph_updates(user_input: str):
    """Stream responses from the graph for real-time output."""
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

In [ ]:
stream_graph_updates("what is portland known for?")

## Structured Output with Pydantic

Applications often need LLM responses in parseable formats (e.g., JSON) for downstream processing. NVIDIA NIM supports structured generation using guided JSON schemas. We use Pydantic's `BaseModel` to define the expected output structure. The `Literal` type restricts the output to specific values.

In [ ]:
from pydantic import BaseModel
from typing import Literal

class UserIntent(BaseModel):
    """The user's current intent in the conversation"""
    intent: Literal["naruto", "bleach"]

Reference: [NIM Structured Generation Docs](https://docs.nvidia.com/nim/large-language-models/latest/structured-generation.html)

In [ ]:
llm_structured = init_chat_model(model=MODEL_ID, model_provider="nvidia").with_structured_output(UserIntent, strict=True)

In [ ]:
# llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(CONTAINER_PORT), model=MODEL_ID).with_structured_output(UserIntent, strict=True)

Use `.with_structured_output()` to enforce the Pydantic schema on LLM responses.

In [ ]:
# Test: Classify user intent based on anime question
res = llm_structured.invoke([
    {'role':'system','content':'You are an anime encyclopedia. Classify if the user is asking a question on naruto or bleach.'},
    {'role':'user','content':'who is sasuke?'}
])

In [ ]:
print(f'intent: {res}')

## Memory

AI applications need memory to share context across multiple interactions.

In LangGraph, you can add two types of memory:
1) Short term memory (thread-level persistence) - this enables agents to track multi-turn conversations.
2) Long term memory - use this to store user-specific or application-specific data across conversations.

We will only utilise short term memory in this tutorial & challenge.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  
from langgraph.graph import StateGraph
import json
from langchain_core.messages import convert_to_openai_messages

checkpointer = InMemorySaver()  

graph = graph_builder.compile(checkpointer=checkpointer)  

res = graph.invoke(
    {"messages": [{"role": "user", "content": "what is cuda?"}]},
    {"configurable": {"thread_id": "1"}},
    
)

In [ ]:
json.dumps(convert_to_openai_messages(res['messages']))

Expected output:

```json
[{"role": "user", "content": "what is cuda?"}, {"role": "assistant", "content": "CUDA (Parallel Computation Engine) is a parallel computing platform and programming model developed by NVIDIA. It allows developers to harness the power of multiple graphics processing units ("}]
```

In [ ]:
res = graph.invoke(
    {"messages": [{"role": "user", "content": "what was my previous question?"}]},
    {"configurable": {"thread_id": "1"}},  
)

In [ ]:
json.dumps(convert_to_openai_messages(res['messages']))

Expected output:

```json
[{"role": "user", "content": "what is cuda?"}, {"role": "assistant", "content": "CUDA (Parallel Computation Engine) is a parallel computing platform and programming model developed by NVIDIA. It allows developers to harness the power of multiple graphics processing units ("}, {"role": "user", "content": "what was my previous question?"}, {"role": "assistant", "content": "Your previous question was: \\"what is cuda?\\""}]
```

By running `graph.invoke` with the same thread id, i.e. ` {"configurable": {"thread_id": "1"}}`, the graph keeps track of previous conversations and is able to utilise its history to continue the conversation

## Interrupts

Interrupts allow you to pause graph execution at specific points and wait for external input before continuing.  
This enables human-in-the-loop patterns where you need external input to proceed.  
When an interrupt is triggered, LangGraph saves the graph state using its persistence layer and waits indefinitely until you resume execution.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt

class FormState(TypedDict):
    age: int | None

def dummy_start_node(state: FormState):
    print('start node')

def get_age_node(state: FormState):
    prompt = "What is your age?"

    while True:
        answer = interrupt(prompt)  # payload surfaces in result["__interrupt__"]

        if isinstance(answer, int) and answer > 0:
            return {"age": answer}

        prompt = f"'{answer}' is not a valid age. Please enter a positive number."

memory = InMemorySaver()

builder = StateGraph(FormState)
builder.add_node("dummy_start_node",dummy_start_node)
builder.add_node("collect_age", get_age_node)
builder.add_edge(START,"dummy_start_node")
builder.add_edge("dummy_start_node","collect_age")
builder.add_edge("collect_age", END)

graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "form-1"}}
first = graph.invoke({"age": None}, config=config)
print(first["__interrupt__"])  # -> [Interrupt(value='What is your age?', ...)]

Expected output:

```
start node.  
[Interrupt(value='What is your age?', id='93589a0b323f03eeaa19f89000f5216c')].  
```

The graph starts with `dummy_start_node` which prints 'start node'.

In `graph.invoke({"age": None}, config=config)`, `{"age": None}` is passed as the state in the graph. This returns an interrupt of 'What is your age?"


In [ ]:
# Provide invalid data; the node re-prompts
retry = graph.invoke(Command(resume="thirty"), config=config)
print(retry["__interrupt__"])  # -> [Interrupt(value="'thirty' is not a valid age...", ...)]

Expected output:

```
[Interrupt(value="'thirty' is not a valid age. Please enter a positive number.", id='93589a0b323f03eeaa19f89000f5216c')]. 
```

In `graph.invoke(Command(resume="thirty"), config=config)`, the state still retains the value of the initial invocation, i.e. `{"age": None}`; the value "thirty" in `Command(resume="thirty")` is returned to the variable 'answer' in `get_age_node`.

In [ ]:
# Provide valid data; loop exits and state updates
final = graph.invoke(Command(resume=30), config=config)
print(final["age"])  # -> 30

Expected output:

```
30
```

In `graph.invoke(Command(resume=30), config=config)`, the state does not change as above. The value '30' in `Command(resume=30)` is returned to the variable 'answer' in `get_age_node` and this is returned as the final value without an interrupt.

<b>It is important to note that when running `graph.invoke(Command(resume=30), config=config)`, the node that raised an interrupt is rerun entirely; thus everything in the function `get_age_node` gets rerun from `prompt = "What is your age?"` each time.</b>

## Agent Skills

Agent skills are sets of instructions, scripts, and resources that agents can discover and load dynamically to perform better at specific tasks.  
At its core, a skill is a folder containing a SKILL.md file. At a minimum, the skill.md file should contain 'name' and 'description' metadata fields. It can also contain instructions specifying the capabilities of the skill.

Skills use progressive disclosure to manage context efficiently.
* Discovery: At startup, agents load only the name and description of each available skill, just enough to know when it might be relevant.
* Activation: When a task matches a skill’s description, the agent reads the instructions of the skill.
* Execution: The agent follows the instructions found in the skill.

Compared to the MCP protocol where the entire input schema has to be part of the agent's context right from the beginning, only the name and description of each skill is fed into the agent's context at the start and instructions are only loaded on a as needed basis.

Skills can also optionally include scripts, references and assets. We'll skip these for the purpose of this tutorial & challenge.

### Skills Implementation

There are [2 main ways to integrate skills into your agent](https://agentskills.io/integrate-skills).

1. filesystem based agents

Operate within a computer environment (bash/unix) and represent the most capable option. Skills are activated when models issue shell commands like cat /path/to/my-skill/SKILL.md. Bundled resources are accessed through shell commands.

2. tool based agents

Function without a dedicated computer environment. Instead, they implement tools allowing models to trigger skills and access bundled assets. The specific tool implementation is up to the developer.

For the purpose of this tutorial & challenge, we're focusing on filesystem based agents. Refer to LangChain's docs for the implementation of [tool based agents](https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant).

Create the `skills` folder. This will serve as the directory for all skills. We will only work with 1 skill for this tutorial - `sales-analytics`.  
This references the [skills sql assistant](https://github.com/langchain-ai/docs/blob/77684f277bb731e35a58dff38f1295a69fb389ad/src/oss/langchain/multi-agent/skills-sql-assistant.mdx).

In [ ]:
!mkdir -p skills/sales-analytics

Following the [standard specification for skills](https://agentskills.io/specification), we

1. Populate the name and description in the frontmatter. 
2. Fill up the instructions after the frontmatter. [optional]

Following the [guidance](https://agentskills.io/specification#progressive-disclosure), the name and description fields makes up approximately 100 tokens while the instructions should be less than 5000 tokens.

Standard SKILL.md template
```
---
name: skill-name
description: A description of what this skill does and when to use it.
---
<instructions in markdown here>
```


In [ ]:
%%writefile skills/sales-analytics/SKILL.md
---
name: sales-analytics
description: Database schema and business logic for sales data analysis including customers, orders, and revenue.
---
# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active customers**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**Revenue calculation**: Only count orders with status = 'completed'. Use total_amount from orders table, which already accounts for discounts.

**Customer lifetime value (CLV)**: Sum of all completed order amounts for a customer.

**High-value orders**: Orders with total_amount > 1000

## Example Query

-- Get top 10 customers by revenue in the last quarter
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10

Import the necessary libraries.  `skills_ref` includes the helper functions that lists, validates and parses agent skills. This is based on the official [repo](https://github.com/agentskills/agentskills/tree/main) released by Anthropic.

In [ ]:
import subprocess
from typing import NotRequired
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware, AgentState
from langchain.messages import SystemMessage
from typing import Callable
from pathlib import Path
from skills_ref.utils import list_skills
from skills_ref.models import SkillProperties

Define the state used by agents to store skills metadata 

In [ ]:
class SkillsState(AgentState):
    """State for the skills middleware."""

    skills_metadata: NotRequired[list[SkillProperties]]
    """List of loaded skill metadata (name, description)."""

Create a bash command skill that allows the agent to list directories, read files and execute sql queries.

In [ ]:
@tool
def bash(command: str) -> str:
    """Execute a bash command and return the output.

    Use this to access skills and resources on the filesystem:
    - Read skill files: cat /path/to/skill/SKILL.md
    - List directories: ls /path/to/dir

    Args:
        command: The bash command to execute
    """
    if not command.startswith(('ls','cat','sqlite3')):
        return 'only "ls","cat" and "sqlite3" commands are allowed'
    result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
    output = result.stdout
    if result.returncode != 0 and result.stderr:
        output = f"Error (exit {result.returncode}): {result.stderr}\n{output}"
    return output or result.stderr

<table style="width: 100%; table-layout: fixed;">
  <tr>
    <td style="width: 50%; text-align: center; vertical-align: top; padding: 10px;">
      <div>The core agent loop involves calling a model, letting it choose tools to execute, and then finishing when it calls no more tools.</div>
    </td>
    <td style="width: 50%; text-align: center; vertical-align: top; padding: 10px;">
      <div>Middleware exposes hooks before and after each of those steps.</div>
    </td>
  </tr>
  <tr>
     <td style="width: 50%; text-align: center; vertical-align: top; padding: 10px;">
      <img src="./images/core_agent_loop.jpeg" alt="core agent loop" style="max-width: 100%; height: auto; display: block; margin: 10px auto;">
    </td>
    <td style="width: 50%; text-align: center; vertical-align: top; padding: 10px;">
      <img src="./images/middleware_final.jpeg" alt="middleware agent loop" style="max-width: 50%; height: auto; display: block; margin: 10px auto;">
    </td>
  </tr>
</table>

Create custom middleware that injects skill descriptions into the system prompt.  
The full list of properties and functions that can be implemented in the Agent Middleware interface can be found [here]((https://github.com/langchain-ai/langchain/blob/c930062f69bbf72d0147db2e2db1940777966ffe/libs/langchain_v1/langchain/agents/middleware/types.py#L343-L756)).  
We are only interested in `state_schema`,`tools`, `before_agent` and `wrap_model_call` for the purpose of this tutorial & challenge.  
`state_schema` contains a list of skill metadata. This is used to expose skills to the model (LLM) through injection of skill prompts.  
We specify `tools = [bash]` to allow the agent to utilize the `bash` tool to list directories, read instructions of the skills and execute sql queries.  
In `before_agent`, we retrieve the metadata(name, description) of the skills dynamically from the system directory and update the agent's `state_schema`. Refer to [utils.py](./skills_ref/utils.py) for more details.  
In `wrap_model_call`, we retrieve the skills metadata from the agent's state, build the skills addendum and append it to the system prompt.

It is important to note that an agent running in production requires additional guardrails for its use of tools.  
Example:
1) Tools should be restricted to running scripts in a constrained folder/environment. 
2) There should be approval steps before running tools that issue write commands.
3) Maintain a list of allowed commands, preferrably only those that are readonly.
4) Implement access controls for sensitive data, e.g. OAuth with Model Context protocol
5) Human in the loop to approve of critical actions

In [ ]:
# Create skill middleware
class SkillMiddleware(AgentMiddleware):
    """Middleware that injects skill descriptions into the system prompt."""

    state_schema = SkillsState

    # Register the bash tool as a class variable
    tools = [bash]

    def __init__(self,skills_dir):
        self.skills_dir = skills_dir

    def before_agent(self, state:SkillsState, runtime):
        skills = list_skills(self.skills_dir)
        return SkillsState(skills_metadata=skills)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Sync: Inject skill descriptions into system prompt."""

        skills = request.state.get("skills_metadata", [])
        skills_list = []
        for skill in skills:
            skills_list.append(
                f"- **{self.skills_dir / skill.name}**: {skill.description}"
            )
        self.skills_prompt = "\n".join(skills_list)

        # Build the skills addendum
        skills_addendum = (
            f"\n\n## Available Skills\n\n{self.skills_prompt}\n\n"
            "Use the bash tool to read skill instructions, e.g., "
            "`bash('cat /path/to/skill/SKILL.md')`. "
        )

        # Append to system message content blocks
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}
        ]
        new_system_message = SystemMessage(content=new_content)
        modified_request = request.override(system_message=new_system_message)
        response = handler(modified_request)
        return response

Define function to create agent - specify the middleware and response format here.

In [ ]:
from pydantic import BaseModel, Field

class SQLOutput(BaseModel):
    sql: str = Field(description="runnable SQL query")

from langchain.chat_models import init_chat_model

def create_sql_agent(skills_dir,inf_url,nvidia_api_key,debug=False):
    model_id = "nvidia/nemotron-3-nano-30b-a3b"
    nvidia_model = init_chat_model(model=model_id,base_url=inf_url,api_key=nvidia_api_key,model_provider="nvidia")
    # Create the agent with skill support
    agent = create_agent(
        nvidia_model,
        system_prompt=(
            f"""
            You are a SQL query assistant that generates runnable SQL query for a sales-analytics database.
            Only return the SQL query and nothing else.
            """
        ),
        middleware=[SkillMiddleware(skills_dir)],
        response_format=SQLOutput,
        debug=debug
    )
    return agent

Specify the skills directory and initialize the agent.

In [ ]:
skills_dir = Path.cwd().resolve() / 'skills'
inf_url = "https://integrate.api.nvidia.com/v1"
nvidia_api_key = os.environ["NVIDIA_API_KEY"]
agent = create_sql_agent(skills_dir,inf_url,nvidia_api_key,debug=True)

Ask a question that triggers skill loading.

In [ ]:
result = agent.invoke(  
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Write a SQL query to find all customers "
                    "who made orders over $1000 in the last month"
                ),
            }
        ]
    }
)

# Print the conversation
for message in result["messages"]:
    if hasattr(message, 'pretty_print'):
        message.pretty_print()
    else:
        print(f"{message.type}: {message.content}")

Expected output:

```
================================ Human Message =================================

Write a SQL query to find all customers who made orders over $1000 in the last month
================================== Ai Message ==================================
Tool Calls:
  bash (chatcmpl-tool-ad22f515e713f986)
 Call ID: chatcmpl-tool-ad22f515e713f986
  Args:
    command: cat /Users/chongenl/Documents/agentic-ai-bootcamp/tutorial/jupyter_notebook/skills/sales-analytics
================================= Tool Message =================================
Name: bash

Error (exit 1): cat: /Users/chongenl/Documents/agentic-ai-bootcamp/tutorial/jupyter_notebook/skills/sales-analytics: Is a directory


================================== Ai Message ==================================
Tool Calls:
  bash (chatcmpl-tool-80378d823d869bdf)
 Call ID: chatcmpl-tool-80378d823d869bdf
  Args:
    command: ls -R /Users/chongenl/Documents/agentic-ai-bootcamp/tutorial/jupyter_notebook/skills/sales-analytics
================================= Tool Message =================================
Name: bash

SKILL.md

================================== Ai Message ==================================
Tool Calls:
  bash (chatcmpl-tool-aaa17cbb062a49c7)
 Call ID: chatcmpl-tool-aaa17cbb062a49c7
  Args:
    command: cat /Users/chongenl/Documents/agentic-ai-bootcamp/tutorial/jupyter_notebook/skills/sales-analytics/SKILL.md
================================= Tool Message =================================
Name: bash

---
name: sales-analytics
description: Database schema and business logic for sales data analysis including customers, orders, and revenue.
---
# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active customers**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**Revenue calculation**: Only count orders with status = 'completed'. Use total_amount from orders table, which already accounts for discounts.

**Customer lifetime value (CLV)**: Sum of all completed order amounts for a customer.

**High-value orders**: Orders with total_amount > 1000

## Example Query

-- Get top 10 customers by revenue in the last quarter
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10

================================== Ai Message ==================================

{
    "sql": "SELECT DISTINCT c.customer_id, c.name, c.email\nFROM customers c\nJOIN orders o ON c.customer_id = o.customer_id\nWHERE o.status = 'completed'\n  AND o.total_amount > 1000\n  AND o.order_date >= CURRENT_DATE - INTERVAL '1 month';"
}
```

Retrieve the structured output.

In [ ]:
print(result['structured_response'].sql)

Expected output:

```
SELECT DISTINCT c.customer_id, c.name, c.email
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.total_amount > 1000
  AND o.order_date >= CURRENT_DATE - INTERVAL '1 month';
```

### Agent skills vs MCP

| Situation                                                  | Prefer skills         | Prefer MCP    | Best combined use                                                                                                                      |
| ---------------------------------------------------------- | --------------------- | ------------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| Need smarter behavior on a task (same tools already exist) | Yes                   | Not necessary | Add skills that tell the agent how to plan, check, and format results using existing tools.                                |
| Need to connect to new systems or data                     | Not sufficient        | Yes           | Define MCP tools for those systems; optionally add skills that describe how to use each tool safely and effectively. |
| Want reusable domain “playbooks” across products           | Yes                   | Optional      | Skills capture the playbooks; MCP is used underneath wherever tools are needed.                                              |
| Enterprise integration, security, auditability are primary | Helpful but secondary | Yes           | MCP enforces permissions; skills encode internal policies and workflows.                                             |

## Links and Resources

- [LangGraph repo](https://github.com/langchain-ai/langgraph)
- [LangGraph short term memory](https://docs.langchain.com/oss/python/langgraph/add-memory#add-short-term-memory)
- [LangGraph Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangChain Agent Skills](https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant)
- [Agent skills open protocol](https://agentskills.io/home)
- [LangChain NVIDIA](https://github.com/langchain-ai/langchain-nvidia)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.

<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="04_low_level_mcp.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_opencode_intro.ipynb">1</a>
        <a href="02_inference_endpoint.ipynb">2</a>
        <a href="03_introduction_mcp.ipynb">3</a>
        <a href="04_low_level_mcp.ipynb">4</a>
        <a >5</a>
        <a href="06_nemo_agent_toolkit.ipynb">6</a>
        <a href="07_challenge.ipynb">7</a>
        <a href="bonus_challenge/08_bonus_challenge.ipynb">8</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="06_nemo_agent_toolkit.ipynb">Next Notebook</a></span>
</div>